# T-E2 AI Agent自动生成股市周报 — 数据可视化

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI_Agent自动生成股市周报 |
| 小组   | 第 09 组 |
| 成员   | 梁志鹏（25210177）、吴璇子（25210264）、黄悦（25210145）、王鹤洋（25210243）、柯骋（25210150）、罗天（25210207）|
| GitHub | https://github.com/liangzhipeng82138-tech/ds2026-G09-ai_agent_stock_weekly_report |
| Pages  | https://liangzhipeng82138-tech.github.io/ds2026-G09-ai_agent_stock_weekly_report/ |
| 日期   | 2026-05-15 |

## 任务说明

本 Notebook 负责生成周报所需的所有可视化图表：
1. 核心指数涨跌幅柱状图
2. 上证指数近5日走势图
3. 各指数振幅雷达图
4. 北向资金净流入柱状图
5. 两市成交额变化趋势图
6. 市场情绪环形图（涨跌家数、涨跌停、赚钱效应）
7. 指数热力图

图表保存至 `output/` 目录。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import os

# 中文字体设置
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

os.makedirs('../output', exist_ok=True)

# 加载清洗后数据
weekly_metrics = pd.read_csv('../data_clean/weekly_index_metrics.csv')
print(f'已加载周度指标: {len(weekly_metrics)} 个指数')

### 1. 核心指数周涨跌幅柱状图

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

indices = weekly_metrics['指数'].tolist()
changes = weekly_metrics['周涨跌幅(%)'].tolist()

# 红涨绿跌
colors = ['#c0392b' if c >= 0 else '#27ae60' for c in changes]

bars = ax.bar(indices, changes, color=colors, width=0.6, edgecolor='white', linewidth=0.5)

# 添加数值标签
for bar, val in zip(bars, changes):
    y_pos = bar.get_height() + 0.1 if val >= 0 else bar.get_height() - 0.3
    ax.text(bar.get_x() + bar.get_width()/2., y_pos, f'{val:+.2f}%',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=11, fontweight='bold')

ax.axhline(y=0, color='gray', linewidth=0.8)
ax.set_title('本周核心指数周涨跌幅对比', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('涨跌幅（%）', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

# 图例
red_patch = mpatches.Patch(color='#c0392b', label='上涨')
green_patch = mpatches.Patch(color='#27ae60', label='下跌')
ax.legend(handles=[red_patch, green_patch], loc='upper right')

plt.tight_layout()
plt.savefig('../output/01_index_change_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print('已保存: output/01_index_change_bar.png')

### 2. 上证指数近5日走势图

In [ ]:
# 加载上证指数清洗后数据
sh_path = '../data_clean/index_上证指数_clean.csv'
if os.path.exists(sh_path):
    sh_df = pd.read_csv(sh_path)
    sh_df['date'] = pd.to_datetime(sh_df['date'])
    
    # 取最近5个交易日
    recent = sh_df.tail(5)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(recent['date'], recent['close'], 'o-', color='#2980b9', 
            linewidth=2.5, markersize=8, markerfacecolor='#2980b9',
            markeredgecolor='white', markeredgewidth=1.5)
    
    # 填充区域
    ax.fill_between(recent['date'], recent['close'], recent['close'].min() - 10,
                     alpha=0.15, color='#2980b9')
    
    # 标注每日收盘价
    for _, row in recent.iterrows():
        ax.annotate(f'{row["close"]:.2f}', 
                    xy=(row['date'], row['close']),
                    xytext=(0, 12), textcoords='offset points',
                    ha='center', fontsize=9, fontweight='bold')
    
    ax.set_title('上证指数近5日走势', fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel('收盘价', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)
    
    # X轴日期格式
    fig.autofmt_xdate()
    
    plt.tight_layout()
    plt.savefig('../output/02_sh_index_trend.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('已保存: output/02_sh_index_trend.png')
else:
    print('未找到上证指数数据')

### 3. 各指数振幅雷达图

In [ ]:
# 雷达图 - 各指数振幅对比
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = weekly_metrics['指数'].tolist()
values = weekly_metrics['振幅(%)'].tolist()

# 闭合雷达图
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
values_closed = values + [values[0]]
angles_closed = angles + [angles[0]]

ax.plot(angles_closed, values_closed, 'o-', linewidth=2, color='#e67e22')
ax.fill(angles_closed, values_closed, alpha=0.25, color='#e67e22')

ax.set_xticks(angles)
ax.set_xticklabels(categories, fontsize=11)
ax.set_title('本周各指数振幅对比（%）', fontsize=16, fontweight='bold', pad=20)

# 添加数值标注
for angle, val in zip(angles, values):
    ax.annotate(f'{val:.2f}%', xy=(angle, val), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.savefig('../output/03_amplitude_radar.png', dpi=300, bbox_inches='tight')
plt.show()
print('已保存: output/03_amplitude_radar.png')

### 4. 北向资金净流入柱状图

In [ ]:
nb_path = '../data_clean/northbound_weekly.csv'
if os.path.exists(nb_path):
    nb_df = pd.read_csv(nb_path)
    nb_df['date'] = pd.to_datetime(nb_df['date'])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    if 'net_flow' in nb_df.columns:
        colors = ['#c0392b' if v >= 0 else '#27ae60' for v in nb_df['net_flow']]
        bars = ax.bar(nb_df['date'].dt.strftime('%m-%d'), nb_df['net_flow'], 
                      color=colors, width=0.5, edgecolor='white', linewidth=0.5)
        
        # 数值标签
        for bar, val in zip(bars, nb_df['net_flow']):
            y_pos = bar.get_height() + 1 if val >= 0 else bar.get_height() - 1
            ax.text(bar.get_x() + bar.get_width()/2., y_pos, f'{val:.1f}',
                    ha='center', va='bottom' if val >= 0 else 'top', fontsize=10)
    
    ax.axhline(y=0, color='gray', linewidth=0.8)
    ax.set_title('北向资金近5日净流入（亿元）', fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel('净流入（亿元）', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    
    red_patch = mpatches.Patch(color='#c0392b', label='净流入')
    green_patch = mpatches.Patch(color='#27ae60', label='净流出')
    ax.legend(handles=[red_patch, green_patch], loc='upper right')
    
    plt.tight_layout()
    plt.savefig('../output/04_northbound_flow.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('已保存: output/04_northbound_flow.png')
else:
    print('未找到北向资金数据')

### 5. 两市成交额变化趋势图

In [ ]:
turnover_path = '../data_raw/turnover.csv'
if os.path.exists(turnover_path):
    turnover = pd.read_csv(turnover_path)
    turnover['date'] = pd.to_datetime(turnover['date'])
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    if '两市合计' in turnover.columns:
        ax.plot(turnover['date'], turnover['两市合计']/1e8, 'o-', 
                color='#16a085', linewidth=2.5, markersize=6,
                markerfacecolor='#16a085', markeredgecolor='white', markeredgewidth=1)
        ax.fill_between(turnover['date'], turnover['两市合计']/1e8, alpha=0.15, color='#16a085')
        
        # 万亿参考线
        ax.axhline(y=10000, color='gray', linestyle='--', linewidth=0.8, label='万亿关口')
    
    ax.set_title('两市成交额变化趋势（亿元）', fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel('成交额（亿元）', fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3)
    ax.legend()
    
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig('../output/05_turnover_trend.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('已保存: output/05_turnover_trend.png')
else:
    print('未找到成交额数据')

### 6. 市场情绪环形图

In [ ]:
sentiment_path = '../data_clean/sentiment.csv'
if os.path.exists(sentiment_path):
    sentiment = pd.read_csv(sentiment_path).iloc[0]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1. 涨跌家数
    rise = int(sentiment['上涨家数'])
    fall = int(sentiment['下跌家数'])
    flat = int(sentiment['平盘家数'])
    axes[0].pie([rise, fall, flat], labels=['上涨', '下跌', '平盘'],
                colors=['#c0392b', '#27ae60', '#95a5a6'],
                autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                wedgeprops=dict(width=0.4, edgecolor='white'))
    axes[0].set_title('涨跌家数分布', fontsize=14, fontweight='bold')
    
    # 2. 涨跌停
    limit_up = int(sentiment['涨停家数'])
    limit_down = int(sentiment['跌停家数'])
    axes[1].pie([limit_up, limit_down], labels=['涨停', '跌停'],
                colors=['#e67e22', '#c0392b'],
                autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                wedgeprops=dict(width=0.4, edgecolor='white'))
    axes[1].set_title('涨跌停分布', fontsize=14, fontweight='bold')
    
    # 3. 赚钱效应
    profit_rate = float(sentiment['赚钱效应(%)'])
    axes[2].pie([profit_rate, 100-profit_rate], labels=['赚钱', '亏钱'],
                colors=['#27ae60', '#c0392b'],
                autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                wedgeprops=dict(width=0.4, edgecolor='white'))
    axes[2].set_title(f'赚钱效应 ({profit_rate:.1f}%)', fontsize=14, fontweight='bold')
    
    plt.suptitle('本周市场情绪分析', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../output/06_sentiment_rings.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('已保存: output/06_sentiment_rings.png')
else:
    print('未找到情绪数据')

### 7. 指数表现热力图

In [ ]:
# 热力图 - 各指数本周表现
fig, ax = plt.subplots(figsize=(14, 4))

# 构建热力图数据
metrics_cols = ['周涨跌幅(%)', '振幅(%)']
heatmap_data = weekly_metrics.set_index('指数')[metrics_cols]

# 归一化用于颜色映射
im = ax.imshow(heatmap_data.values, cmap='RdYlGn', aspect='auto')

# 标签
ax.set_xticks(range(len(metrics_cols)))
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=11)

# 数值标注
for i in range(len(heatmap_data)):
    for j in range(len(metrics_cols)):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_title('本周各指数核心指标热力图', fontsize=16, fontweight='bold', pad=15)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../output/07_index_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('已保存: output/07_index_heatmap.png')
print('\n✓ 所有图表生成完成！')